# Positional Encoding Tekniklkeri

7 Yaklaşım : Sinusoidal, Learned Absolute, Relative, RoPE, ALiBi, ConvPos, NoPE

Aşağıdaki Python kodu, belirttiğiniz yedi farklı konumsal kodlama (Positional Encoding) tekniğini, tek bir örnek metin üzerinden adım adım uygular. Her yöntemin felsefesini, ara adımlarını ve nihai çıktısını net bir şekilde görebilmeniz için bolca açıklama ve print ifadesi içerir.

**Kodun Yapısı ve Anlatımı**

Kod, bir cümleyi işlerken her bir PE tekniğinin ne yaptığını canlandırır. İki ana kategoriye ayrılır:

* Vektör Modifikasyonu Yapanlar (Sinusoidal, Learned, RoPE, ConvPos): Bu yöntemler, kelimelerin embedding vektörlerini, attention mekanizmasına girmeden önce değiştirir.

* Attention Skorunu Modifiye Edenler (ALiBi, Relative/T5): Bu yöntemler, embedding vektörlerine dokunmaz. Bunun yerine, attention mekanizmasının içinde, Query ve Key etkileşiminden doğan skorlara bir "yanlılık" (bias) ekler.

* Baseline (NoPE): Hiçbir şey yapmayan ve karşılaştırma için temel oluşturan durum.

Bu yapı, yöntemler arasındaki temel felsefe farkını anlamak için çok önemlidir.

Senaryo: input_sentence = "yapay zeka geleceği şekillendiriyor" cümlesini ele alacağız ve özellikle "geleceği"

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import time

# --- BÖLÜM 0: KURULUM VE GENEL HAZIRLIK ---

# Genel Ayarlar
D_MODEL = 32  # Anlaşılırlık için embedding boyutu
input_sentence = "yapay zeka geleceği şekillendiriyor"
tokens = input_sentence.split()
SEQ_LEN = len(tokens)
VOCAB = {"<pad>": 0, **{word: i+1 for i, word in enumerate(sorted(list(set(tokens))))}}
token_ids = torch.tensor([VOCAB[word] for word in tokens])

# Modelin temel embedding katmanı
embedding_layer = nn.Embedding(len(VOCAB) + 1, D_MODEL)
word_embeddings = embedding_layer(token_ids)

# Karşılaştırmada odaklanacağımız kelime ve pozisyonu
TARGET_WORD_INDEX = 2
TARGET_WORD = tokens[TARGET_WORD_INDEX]


def print_header(title):
    print("\n" + "="*70)
    print(f"### {title} ###")
    print("="*70)

def print_step(step_title, delay=0.5):
    print(f"\n--- {step_title} ---\n")
    time.sleep(delay)

print_header("7 Farklı Positional Encoding Tekniğinin Karşılaştırmalı Simülasyonu")
print(f"Örnek Cümle: '{input_sentence}'")
print(f"Odak Kelime: '{TARGET_WORD}' (Pozisyon: {TARGET_WORD_INDEX})")
print(f"Vektör Boyutu (d_model): {D_MODEL}")
print_step("Başlangıç: Orijinal Kelime Embedding'leri (Pozisyon Bilgisi Yok)")
print(f"'{TARGET_WORD}' kelimesinin orijinal embedding vektörünün başı:\n{word_embeddings[TARGET_WORD_INDEX, :8].detach().numpy().round(2)}...")


# ==============================================================================
# KATEGORİ 1: VEKTÖR MODİFİKASYONU YAPAN YÖNTEMLER
# ==============================================================================

# --- YÖNTEM 1: Sinusoidal PE (Mutlak, Sabit) ---
print_header("YÖNTEM 1: Sinusoidal Positional Encoding")

def get_sinusoidal_pe(seq_len, d_model):
    pe = torch.zeros(seq_len, d_model)
    position = torch.arange(0, seq_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-np.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

pe_matrix = get_sinusoidal_pe(SEQ_LEN, D_MODEL)
pe_vector = pe_matrix[TARGET_WORD_INDEX]
final_vector_sinusoidal = word_embeddings[TARGET_WORD_INDEX] + pe_vector
print(f"'{TARGET_WORD}' için pozisyon vektörü (özet):\n{pe_vector[:8].numpy().round(2)}...")
print(f"SONUÇ: Pozisyon bilgisi eklenmiş yeni vektör (özet):\n{final_vector_sinusoidal.detach().numpy().round(2)}...")
print("YORUM: Vektöre sabit, matematiksel bir pozisyon imzası eklendi.")


# --- YÖNTEM 2: Learned Absolute PE (Mutlak, Öğrenilmiş) ---
print_header("YÖNTEM 2: Learned Absolute Positional Encoding")
learned_pe_layer = nn.Embedding(SEQ_LEN, D_MODEL)
learned_pe_vector = learned_pe_layer(torch.tensor(TARGET_WORD_INDEX))
final_vector_learned = word_embeddings[TARGET_WORD_INDEX] + learned_pe_vector
print(f"'{TARGET_WORD}' için öğrenilebilir pozisyon vektörü (rastgele) (özet):\n{learned_pe_vector.detach().numpy()[:8].round(2)}...")
print(f"SONUÇ: Pozisyon bilgisi eklenmiş yeni vektör (özet):\n{final_vector_learned.detach().numpy()[:8].round(2)}...")
print("YORUM: Bu pozisyon vektörü, eğitim sırasında veriye göre optimize edilir.")


# --- YÖNTEM 3: Rotary Positional Encoding (RoPE) (Göreceli, Sabit) ---
print_header("YÖNTEM 3: Rotary Positional Encoding (RoPE)")

def apply_rotary_pos_emb(x, pos, d_model):
    angle_rads = 1 / (10000 ** (torch.arange(0, d_model, 2).float() / d_model))
    angles = pos * angle_rads
    cos_val = torch.cos(angles).repeat_interleave(2)
    sin_val = torch.sin(angles).repeat_interleave(2)
    x_rotated_part = torch.cat([-x[1::2], x[0::2]], dim=-1)
    return x * cos_val + x_rotated_part * sin_val

rotated_vector_rope = apply_rotary_pos_emb(word_embeddings[TARGET_WORD_INDEX], TARGET_WORD_INDEX, D_MODEL)
print(f"SONUÇ: Pozisyon bilgisiyle 'döndürülmüş' yeni vektör (özet):\n{rotated_vector_rope.detach().numpy()[:8].round(2)}...")
print("YORUM: Vektöre bir şey eklenmedi, değerleri pozisyona göre değiştirildi (döndürüldü).")


# --- YÖNTEM 4: ConvPos (Yerel/Lokal, Öğrenilmiş) ---
print_header("YÖNTEM 4: Convolutional Positional Encoding (ConvPos)")
# Kernel size 3, yani her kelime kendisi ve sağındaki/solundaki komşusundan etkilenir
conv_layer = nn.Conv1d(in_channels=D_MODEL, out_channels=D_MODEL, kernel_size=3, padding=1)
# Conv1d (Batch, Channels, Length) formatı bekler
conv_input = word_embeddings.unsqueeze(0).permute(0, 2, 1)
conv_output = conv_layer(conv_input).permute(0, 2, 1).squeeze(0)
final_vector_conv = conv_output[TARGET_WORD_INDEX]
print(f"'{TARGET_WORD}' kelimesinin komşuları: '{tokens[TARGET_WORD_INDEX-1]}' ve '{tokens[TARGET_WORD_INDEX+1]}'")
print(f"SONUÇ: Komşularının bilgisiyle harmanlanmış yeni vektör (özet):\n{final_vector_conv.detach().numpy()[:8].round(2)}...")
print("YORUM: Vektör artık sadece kendi anlamını değil, yerel komşularının da bilgisini içeriyor.")


# ==============================================================================
# KATEGORİ 2: ATTENTION SKORUNU MODİFİYE EDEN YÖNTEMLER
# ==============================================================================
print_header("KATEGORİ 2: ATTENTION SKORUNU DEĞİŞTİREN YÖNTEMLER")
# Bu yöntemler için, (Query @ Key.T) sonrası oluşan bir skor matrisini simüle edelim.
dummy_attn_scores = torch.randn(SEQ_LEN, SEQ_LEN)
print("Bu yöntemler, aşağıdaki simüle edilmiş (rastgele) Attention Skor Matrisini değiştirir.")
print(dummy_attn_scores.numpy().round(2))

# --- YÖNTEM 5: ALiBi (Göreceli, Sabit Bias) ---
print_header("YÖNTEM 5: Attention with Linear Biases (ALiBi)")
distances = torch.arange(SEQ_LEN).view(-1, 1) - torch.arange(SEQ_LEN).view(1, -1)
# Slope (eğim) sabiti, genellikle başlık sayısına göre belirlenir. Biz bir sabit seçelim.
slope = -0.5
alibi_bias = torch.abs(distances.float()) * slope
final_scores_alibi = dummy_attn_scores + alibi_bias
print("\nALiBi tarafından hesaplanan 'mesafe cezası' (bias) matrisi:")
print(alibi_bias.numpy().round(2))
print("\nSONUÇ: ALiBi bias'ı eklenmiş yeni attention skorları:")
print(final_scores_alibi.numpy().round(2))
print("YORUM: Uzak kelimelerin skoru (sağ üst/sol alt köşeler) daha fazla düşürüldü.")


# --- YÖNTEM 6: Relative PE (T5 Stili) (Göreceli, Öğrenilmiş Bias) ---
print_header("YÖNTEM 6: Relative PE (T5 Stili)")
num_buckets = 8
relative_bias_table = nn.Embedding(num_buckets, 1)
# Mesafeleri kovalara ayıran basit bir fonksiyon
relative_positions = torch.arange(SEQ_LEN).view(-1, 1) - torch.arange(SEQ_LEN).view(1, -1)
# Basit bir kovalama:
bucket_indices = torch.clamp(relative_positions + num_buckets//2, 0, num_buckets-1)
t5_bias = relative_bias_table(bucket_indices).squeeze(-1)
final_scores_t5 = dummy_attn_scores + t5_bias
print("\nHer mesafe için atanan 'kova' indeksleri:")
print(bucket_indices.numpy())
print("\nÖğrenilebilir bias tablosundan gelen yanlılık değerleri (rastgele):")
print(t5_bias.detach().numpy().round(2))
print("\nSONUÇ: T5 stili bias eklenmiş yeni attention skorları:")
print(final_scores_t5.detach().numpy().round(2))
print("YORUM: ALiBi'den farklı olarak, bu bias değerleri eğitim sırasında öğrenilir.")


# ==============================================================================
# KATEGORİ 3: BASELINE
# ==============================================================================

# --- YÖNTEM 7: NoPE (No Positional Encoding) ---
print_header("YÖNTEM 7: No Positional Encoding (NoPE)")
print("Bu yaklaşımda, pozisyon bilgisi eklemek için hiçbir işlem yapılmaz.")
print(f"SONUÇ: '{TARGET_WORD}' kelimesinin vektörü değişmeden kalır (özet):\n{word_embeddings[TARGET_WORD_INDEX, :8].detach().numpy().round(2)}...")
print("YORUM: Bu, Transformer'ın 'doğal' halidir ve kelimelerin sırasını ayırt edemez.")


# ==============================================================================
# FİNAL ÖZET
# ==============================================================================
print_header("FİNAL ÖZET VE KARŞILAŞTIRMA")
print("""
Bu 7 yöntem, pozisyon bilgisini modellemeye yönelik farklı felsefeleri temsil eder:

1.  MUTLAK VE SABİT (Sinusoidal): "Konum evrensel bir kuraldır, matematiksel olarak öğretelim."
    - Avantaj: Verimli, mükemmel genelleme. Dezavantaj: Esnek değil.

2.  MUTLAK VE ÖĞRENİLMİŞ (Learned): "Konumun en iyi temsilini veriden öğrenelim."
    - Avantaj: Esnek. Dezavantaj: Genellemesi zayıf, maliyetli.

3.  GÖRECELİ VE DÖNER (RoPE): "Konumu, vektörlerin doğal etkileşimine içsel bir özellik olarak kodlayalım."
    - Avantaj: Zarif, parametresiz, mükemmel genelleme. Modern modellerin tercihi.

4.  YEREL VE KONVOLÜSYONEL (ConvPos): "Konumu, kelimenin yakın komşularıyla olan ilişkisi olarak öğretelim."
    - Avantaj: Dilin yerel yapısını iyi yakalar.

5.  GÖRECELİ VE SABİT BİAS (ALiBi): "Uzak kelimeler daha az önemlidir, bu kuralı skorlara basitçe ekleyelim."
    - Avantaj: Çok basit, etkili, mükemmel genelleme.

6.  GÖRECELİ VE ÖĞRENİLMİŞ BİAS (T5 Stili): "Uzaklığın etkisini de veriden öğrenelim."
    - Avantaj: Esnek bir bias sağlar. Dezavantaj: Ek parametreler ve karmaşıklık.
    
7.  POZİSYONSUZ (NoPE): "Pozisyonu tamamen yok sayalım." (Bazı özel mimarilerde kullanılır)
    - Bu bizim temel karşılaştırma noktamızdır.
""")